# Imports

In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets

import spacy
from sklearn.feature_extraction.text import TfidfVectorizer

# Preprocessing


## Combining Datasets
The data being used consists of 5 separate datasets all containing 2000 each. Our first step was to combine them into one large one. We also dropped the category column at this point to prevent the model from clustering with it.

In [2]:
bussiness_df = pd.read_csv("business_data.csv")
education_df = pd.read_csv("education_data.csv")
entertainment_df = pd.read_csv("entertainment_data.csv")
sports_df = pd.read_csv("sports_data.csv")
tech_df = pd.read_csv("technology_data.csv")

In [3]:
df = pd.concat([bussiness_df,education_df,entertainment_df,sports_df,tech_df],axis=0)
df.head()

,headlines,description,content,url,category
0,Nirmala Sitharaman to equal Morarji Desai’s re...,With the presentation of the interim budget on...,"Sitharaman, the first full-time woman finance ...",https://indianexpress.com/article/business/bud...,business
1,"‘Will densify network, want to be at least no....","'In terms of market share, we aim to double it...",The merger of Tata group’s budget airlines Air...,https://indianexpress.com/article/business/avi...,business
2,Air India group to induct an aircraft every si...,Air India currently has 117 operational aircra...,The Air India group plans to induct one aircra...,https://indianexpress.com/article/business/avi...,business
3,Red Sea woes: Exporters seek increased credit ...,Rising attacks forced shippers to consider the...,Indian exporters have asked the central govern...,https://indianexpress.com/article/business/red...,business
4,Air India group to induct a plane every 6 days...,"Apart from fleet expansion, 2024 will also see...",The Air India group plans to induct one aircra...,https://indianexpress.com/article/business/avi...,business


In [4]:
df.drop(["url","category"], axis=1,inplace=True)
df.head()

,headlines,description,content
0,Nirmala Sitharaman to equal Morarji Desai’s re...,With the presentation of the interim budget on...,"Sitharaman, the first full-time woman finance ..."
1,"‘Will densify network, want to be at least no....","'In terms of market share, we aim to double it...",The merger of Tata group’s budget airlines Air...
2,Air India group to induct an aircraft every si...,Air India currently has 117 operational aircra...,The Air India group plans to induct one aircra...
3,Red Sea woes: Exporters seek increased credit ...,Rising attacks forced shippers to consider the...,Indian exporters have asked the central govern...
4,Air India group to induct a plane every 6 days...,"Apart from fleet expansion, 2024 will also see...",The Air India group plans to induct one aircra...


In [5]:
print(df.shape,"\n")


(10000, 3) 



In [6]:
df.isna().sum()

headlines      0
description    0
content        0
dtype: int64

## Text Cleaning

* Lowercasing — obvious, but easy to forget
* Removing punctuation, special characters, HTML/URLs depending on your data source
* Stopword removal 
* Lemmatization


In [7]:
def cleanColumn(df, col):
    df[col] = df[col].str.lower()
    
    #Load the model, convert to lemmas, join into one text
    nlp = spacy.load("en_core_web_sm")
    df[col] = df[col].apply(lambda x: nlp(x))
    df[col] = df[col].apply(lambda x: [token.lemma_ for token in x])
    df[col] = df[col].apply(lambda x: " ".join(x))


    #Removing punctuation
    df[col] = df[col].str.replace(".","")
    df[col] = df[col].str.replace(",","")
    df[col] = df[col].str.replace("!","")
    df[col] = df[col].str.replace("?","")
    df[col] = df[col].str.replace(";","")
    df[col] = df[col].str.replace(":", "")
    df[col] = df[col].str.replace("\n", "")

    return df[col]

In [8]:
df2 = df.copy()
for col in df.columns:
    df2[col] = cleanColumn(df2, col)


In [9]:
df2.head()

,headlines,description,content
0,nirmala sitharaman to equal morarji desai ’s r...,with the presentation of the interim budget on...,sitharaman the first full - time woman financ...
1,' will densify network want to be at least no...,' in term of market share we aim to double it...,the merger of tata group ’s budget airlines ai...
2,air india group to induct an aircraft every si...,air india currently have 117 operational aircr...,the air india group plan to induct one aircraf...
3,red sea woe exporter seek increase credit as ...,rise attack force shipper to consider the long...,indian exporter have ask the central governmen...
4,air india group to induct a plane every 6 day ...,apart from fleet expansion 2024 will also see...,the air india group plan to induct one aircraf...


In [ ]:
df2["Text"]  = df2["headlines"] + " " + df2["description"] + " " + df2["content"]
text = df2["Text"].to_list()
text

['nirmala sitharaman to equal morarji desai ’s record with her sixth straight budget with the presentation of the interim budget on february 1  nirmala sitharaman will surpass the record of her predecessor like manmohan singh  arun jaitley  p chidambaram  and yashwant sinha  who have present five budget in a row  sitharaman  the first full - time woman finance minister of the country  have present five full budget since july 2019 and will present an interim or vote - on - account budget next week   with the presentation of the interim budget on february 1  sitharaman will surpass the record of her predecessor like manmohan singh  arun jaitley  p chidambaram  and yashwant sinha  who have present five budget in a row   desai  as finance minister  have present five annual budget and one interim budget between 1959 - 1964  the interim budget 2024 - 25 to be present by sitharaman on february 1  will be a vote - on - account that will give the government authority to spend certain sum of mon

In [ ]:
#Ensuring we still have the correct number of entries
len(text)

10000